In [0]:
# ⚙️ Global Toggles for Pipeline Behavior# These flags allow fine-grained control over the ingestion pipeline# 🧠 GPT Logic ControlsENABLE_GPT_CACHING = False         # Enable/disable schema caching for repeated headersENABLE_GPT_ENRICHMENT = True       # Use GPT to enrich column descriptions (binders glossary)ENABLE_LANGCHAIN_GPT = False       # Use LangChain-based schema parsing (fallback to OpenAI if False)# 🚀 Pipeline TogglesUSE_SEMANTIC_FILTERING = True      # Filter by GPT semantic scoreENABLE_HEADER_SHIFTING = True      # Auto-detect left padding/misaligned headersENABLE_COLUMN_PADDING = True       # Pad shorter rows to match header lengthENABLE_EXTRA_FIELD_CAPTURE = False # Capture spillover/extra fieldsENABLE_DELTA_WRITE = False         # Enable writing to Delta tablesENABLE_QUALITY_CHECK = True        # Enable data quality checksENABLE_MONITORING = True           # Enable performance monitoringENABLE_INSURANCE_VALIDATION = True # Enable insurance-specific validation# 🏥 Insurance-Specific FeaturesENABLE_INSURANCE_TABLE_DETECTION = True  # Use AI to detect multiple tables in bordereauxENABLE_INSURANCE_SCHEMA_INFERENCE = True  # Use insurance-specific schema inference# 📊 ThresholdsSEMANTIC_SCORE_THRESHOLD = 0.2     # Minimum score for semantic column filteringNULL_THRESHOLD = 0.8               # Threshold for null percentage warningsMAX_RETRIES = 3                    # Maximum number of retries for operations# 📁 PathsBASE_DELTA_PATH = "/Volumes/bdx/data_dictionary/bdx_files"  # Base path for Delta tables# 🧠 Cachinggpt_schema_cache = {}# Import necessary libraries for enhanced functionalityimport timefrom datetime import datetimefrom typing import Dict, List, Tuple, Any, Optional

In [0]:
# Performance Monitoring Classclass PerformanceMonitor:    """Simple performance monitoring for the BDX processing pipeline"""        def __init__(self):        self.start_times = {}        self.durations = {}        self.counts = {}        self.logs = []            def start(self, operation):        """Start timing an operation"""        self.start_times[operation] = time.time()        self.log(f"Started: {operation}")            def end(self, operation, count=1):        """End timing an operation and record stats"""        if operation in self.start_times:            duration = time.time() - self.start_times[operation]                        if operation not in self.durations:                self.durations[operation] = []                self.counts[operation] = []                            self.durations[operation].append(duration)            self.counts[operation].append(count)                        self.log(f"Completed: {operation} in {duration:.2f}s (processed {count} items)")            return duration        return None        def log(self, message):        """Add a log entry with timestamp"""        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]        self.logs.append(f"[{timestamp}] {message}")            def get_stats(self):        """Get performance statistics"""        stats = {}                for op in self.durations:            total_duration = sum(self.durations[op])            total_count = sum(self.counts[op])            avg_duration = total_duration / len(self.durations[op])                        stats[op] = {                "total_duration": round(total_duration, 2),                "avg_duration": round(avg_duration, 2),                "calls": len(self.durations[op]),                "items_processed": total_count,                "avg_time_per_item": round(total_duration / total_count, 4) if total_count > 0 else 0            }                    return stats        def print_report(self):        """Print a performance report"""        stats = self.get_stats()                print("\n📊 Performance Report")        print("=" * 80)        print(f"{'Operation':<30} {'Total Time':<12} {'Avg Time':<12} {'Calls':<8} {'Items':<8} {'Time/Item':<12}")        print("-" * 80)                for op, data in sorted(stats.items(), key=lambda x: x[1]['total_duration'], reverse=True):            print(f"{op:<30} {data['total_duration']:<12.2f}s {data['avg_duration']:<12.2f}s {data['calls']:<8} {data['items_processed']:<8} {data['avg_time_per_item']:<12.4f}s")                return stats# Initialize the monitorperformance_monitor = PerformanceMonitor()

In [0]:
# Advanced Table Detection for Insurance Bordereaux Filesdef detect_insurance_tables_with_ai(raw_text):    """    Insurance-specific table detection that handles the unique challenges of bordereaux files.        This function specifically looks for:    - Multiple header rows (often 2-4 rows in insurance bordereaux)    - Footer rows with totals or comments    - Embedded metadata and notes    - Section breaks between different data tables    """    performance_monitor.start("table_detection")        prompt = f"""    You are an expert in insurance bordereaux files analysis.        Insurance bordereaux files typically have these challenging characteristics:    - Multiple header rows (often 2-4 rows)    - Footer rows with totals    - Embedded metadata and notes    - Section breaks between different data tables        Analyze this content and identify:    1. The row ranges for each logical table (start_row, end_row)    2. The header rows for each table (which rows contain headers)    3. Any footer rows to exclude (totals, notes, etc.)        Content:    {raw_text[:4000]}        Respond with a JSON object containing the identified tables.    """        try:        response = client.chat.completions.create(            model=deployment,            messages=[{"role": "user", "content": prompt}],            temperature=0.3        )                # Parse the response to extract table boundaries        result = json.loads(response.choices[0].message.content)        performance_monitor.end("table_detection")        return result    except Exception as e:        print(f"❌ Error in table detection: {e}")        performance_monitor.end("table_detection")        # Return a default structure as fallback        return {            "tables": [                {                    "start_row": 0,                    "end_row": len(raw_text.split("\n")),                    "header_rows": [0],                    "footer_rows": []                }            ]        }def infer_insurance_schema(raw_text):    """    Infer schema specifically for insurance bordereaux data.        This function:    - Identifies common insurance data fields    - Maps variations to standardized names    - Infers appropriate data types for insurance fields    - Assigns semantic meaning to columns    """    performance_monitor.start("schema_inference")        prompt = f"""    You are an expert in insurance bordereaux files analysis.        Analyze this insurance bordereaux content and identify:    1. Column headers and their standardized names    2. Appropriate data types for each column    3. Semantic meaning of each column in insurance context        Insurance bordereaux typically contain these types of fields:    - Policy information (policy number, effective date, expiry date)    - Premium information (gross premium, net premium, commission)    - Risk information (sum insured, limit, deductible)    - Classification (risk code, class of business)    - Insured details (name, location)    - Claims information (if a claims bordereaux)        Content:    {raw_text[:4000]}        Respond with a JSON object containing the column definitions.    """        try:        if ENABLE_LANGCHAIN_GPT:            # Use LangChain for schema inference            chain = get_langchain_gpt_schema_chain(client)            result = chain.run({"prompt_text": raw_text, "context": ""})        else:            # Use direct OpenAI API call            response = client.chat.completions.create(                model=deployment,                messages=[{"role": "user", "content": prompt}],                temperature=0.3            )            result = json.loads(response.choices[0].message.content)                performance_monitor.end("schema_inference")        return result    except Exception as e:        print(f"❌ Error in schema inference: {e}")        performance_monitor.end("schema_inference")        # Return a minimal schema as fallback        return {            "columns": [],            "column_headers": [],            "data_start_row": 0        }

In [0]:
# Insurance-Specific Terminology Standardizationdef standardize_insurance_headers(headers):    """    Standardize header names based on insurance terminology.        This function maps common variations of insurance terms to standardized versions:    - "Gross Premium" / "GP" / "Premium (Gross)" → "Gross_Premium"    - "Net Premium" / "NP" / "Premium (Net)" → "Net_Premium"    - "Commission %" / "Comm %" / "Commission Rate" → "Commission_Rate"    """    # Load insurance terminology mapping    insurance_terms = {        "premium": "Premium",        "gross premium": "Gross_Premium",        "gp": "Gross_Premium",        "premium (gross)": "Gross_Premium",        "net premium": "Net_Premium",        "np": "Net_Premium",        "premium (net)": "Net_Premium",        "commission": "Commission",        "commission %": "Commission_Rate",        "comm %": "Commission_Rate",        "commission rate": "Commission_Rate",        "broker": "Broker",        "cedant": "Cedant",        "insured": "Insured",        "policy": "Policy",        "policy number": "Policy_Number",        "policy no": "Policy_Number",        "policy #": "Policy_Number",        "effective date": "Effective_Date",        "inception date": "Effective_Date",        "expiry date": "Expiry_Date",        "expiration date": "Expiry_Date",        "limit": "Limit",        "sum insured": "Sum_Insured",        "si": "Sum_Insured",        "deductible": "Deductible",        "excess": "Deductible",        "treaty": "Treaty",        "treaty year": "Treaty_Year",        "ty": "Treaty_Year",        "underwriting year": "Underwriting_Year",        "uw year": "Underwriting_Year",        "uy": "Underwriting_Year",        "risk code": "Risk_Code",        "class of business": "Business_Class",        "business class": "Business_Class",        "lob": "Business_Class",  # Line of Business        "line of business": "Business_Class",        "claim": "Claim",        "claims": "Claims",        "loss ratio": "Loss_Ratio",        "lr": "Loss_Ratio",        "technical result": "Technical_Result",        "tr": "Technical_Result",        "bordereaux": "Bordereaux",        "bdx": "Bordereaux",        "quarter": "Quarter",        "q1": "Q1",        "q2": "Q2",        "q3": "Q3",        "q4": "Q4"    }        standardized = []    for header in headers:        header_lower = header.lower()                # Check for exact matches        if header_lower in insurance_terms:            standardized.append(insurance_terms[header_lower])            continue                    # Check for partial matches        matched = False        for term, replacement in insurance_terms.items():            if term in header_lower:                # Replace only the matching part                new_header = header_lower.replace(term, replacement)                # Convert to snake_case                new_header = "_".join([part.capitalize() for part in new_header.split()])                standardized.append(new_header)                matched = True                break                        if not matched:            # Convert to snake_case if no match            new_header = "_".join([part.capitalize() for part in header.split()])            standardized.append(new_header)        return standardizeddef process_insurance_headers(df, header_rows):    """    Process multi-row headers common in insurance bordereaux files.        This function:    - Combines multiple header rows intelligently    - Handles merged cells by detecting and combining related headers    - Standardizes header names based on insurance terminology    """    # Extract the header rows    headers = df.iloc[header_rows].values.tolist()        # Process merged cells (cells with NaN in subsequent rows)    processed_headers = []    for col_idx in range(len(headers[0])):        col_values = [row[col_idx] for row in headers]        # Filter out NaN and empty values        col_values = [str(val).strip() for val in col_values if pd.notna(val) and str(val).strip()]                if col_values:            # Combine the header values            combined_header = " ".join(col_values)            processed_headers.append(combined_header)        else:            # Use a placeholder for empty headers            processed_headers.append(f"Column_{col_idx}")        # Standardize headers using insurance terminology    standardized_headers = standardize_insurance_headers(processed_headers)        return standardized_headers

In [0]:
# Safe Delta Write Function with Retriesdef safe_delta_write(df, path, table_name=None, mode="overwrite"):    """    Safely write DataFrame to Delta with retries and proper error handling.        Args:        df: Spark DataFrame to write        path: Path to write to        table_name: Optional table name to save as        mode: Write mode (overwrite, append, etc.)            Returns:        bool: Success status    """    performance_monitor.start(f"delta_write_{path}")        for attempt in range(MAX_RETRIES):        try:            if table_name:                df.write.format("delta").mode(mode).saveAsTable(table_name)                print(f"✅ Successfully wrote to table: {table_name}")            else:                df.write.format("delta").mode(mode).save(path)                print(f"✅ Successfully wrote to path: {path}")                            performance_monitor.end(f"delta_write_{path}")            return True        except Exception as e:            if attempt < MAX_RETRIES - 1:                wait_time = 2 ** attempt  # Exponential backoff                print(f"⚠️ Write attempt {attempt+1} failed: {e}. Retrying in {wait_time}s...")                time.sleep(wait_time)            else:                print(f"❌ Failed to write after {MAX_RETRIES} attempts: {e}")                performance_monitor.end(f"delta_write_{path}")                return False

In [0]:
# Secure API Key Managementfrom openai import AzureOpenAIimport osimport jsonimport pandas as pdendpoint = "https://hanna-m9ic9273-eastus2.cognitiveservices.azure.com/"model_name = "gpt-4.1"deployment = "gpt-4.1"# Secure API key managementtry:    # Try to get from Databricks secrets first    subscription_key = dbutils.secrets.get("azure-openai", "api-key")    print("✅ Using API key from Databricks secrets")except Exception as e:    # Fallback to hardcoded key with warning    print(f"⚠️ Warning: Using hardcoded API key. Consider moving to Databricks secrets: {str(e)}")    subscription_key = "C9oe9lxteRbZXvRHdkXRq7uezqsl1bDWQ7tJH91uAHaWbjnpoQ8rJQQJ99BDACHYHv6XJ3w3AAAAACOGb14W"api_version = "2024-12-01-preview"client = AzureOpenAI(    api_version=api_version,    azure_endpoint=endpoint,    api_key=subscription_key,)

In [0]:
# Enhanced Data Quality Validationfrom pyspark.sql.functions import col, isnandef validate_insurance_data_quality(df, sheet_key, null_threshold=NULL_THRESHOLD):    """    Validate data quality specifically for insurance bordereaux data.        This function checks:    - Required fields for insurance compliance    - Date format consistency    - Premium and claim amount validation    - Currency consistency    - Policy number format validation    """    issues = []    row_count = df.count()        if row_count == 0:        return [{            "sheet": sheet_key,            "issue": "❌ Empty DataFrame",            "severity": "high"        }]        # Required fields for insurance bordereaux    required_fields = [        "Policy_Number", "Effective_Date", "Expiry_Date",         "Premium", "Commission", "Insured", "Risk_Code"    ]        # Check for missing required fields    for field in required_fields:        matching_fields = [col for col in df.columns if field.lower() in col.lower()]        if not matching_fields:            issues.append({                "sheet": sheet_key,                "issue": f"❌ Missing required field: {field}",                "severity": "high"            })        # Check for overall data quality    try:        # 1. Check for null values in each column        for field in df.schema.fields:            name = field.name            try:                # Handle both null and NaN values                null_count = df.filter(col(name).isNull() | isnan(col(name))).count()                null_pct = null_count / row_count if row_count > 0 else 0                                if null_pct >= null_threshold:                    issues.append({                        "sheet": sheet_key,                        "column": name,                        "issue": f"⚠️ {int(null_pct * 100)}% nulls",                        "severity": "medium" if null_pct < 0.95 else "high"                    })                                # 2. Check for numeric values stored as strings                if field.dataType.simpleString() == "string":                    # Sample the first 100 rows for efficiency                    sample = df.select(col(name)).limit(100).rdd.flatMap(lambda x: [x[0]]).collect()                    numeric_count = sum(1 for val in sample if val and isinstance(val, str) and                                        val.replace(',', '').replace('.', '').replace('-', '').isdigit())                                        if numeric_count > len(sample) * 0.5:                        issues.append({                            "sheet": sheet_key,                            "column": name,                            "issue": "⚠️ Likely numeric values stored as strings",                            "severity": "low"                        })                                    except Exception as col_error:                issues.append({                    "sheet": sheet_key,                    "column": name,                    "issue": f"❌ Error checking column: {str(col_error)}",                    "severity": "medium"                })                        # 3. Check for duplicate rows        try:            dup_count = row_count - df.dropDuplicates().count()            if dup_count > 0:                issues.append({                    "sheet": sheet_key,                    "issue": f"⚠️ {dup_count} duplicate rows detected",                    "severity": "medium"                })        except Exception as e:            issues.append({                "sheet": sheet_key,                "issue": f"❌ Error checking duplicates: {str(e)}",                "severity": "low"            })                except Exception as e:        issues.append({            "sheet": sheet_key,            "issue": f"❌ Error during quality check: {str(e)}",            "severity": "high"        })            return issuesdef flag_data_quality_issues(df, sheet_key, threshold_null_pct=NULL_THRESHOLD):    """    Enhanced data quality checks with better error handling and more comprehensive checks.        Args:        df: Spark DataFrame to check        sheet_key: Identifier for the sheet        threshold_null_pct: Threshold for null percentage warnings            Returns:        list: List of quality issues found    """    # If insurance validation is enabled, use the more comprehensive validation    if ENABLE_INSURANCE_VALIDATION:        return validate_insurance_data_quality(df, sheet_key, threshold_null_pct)        issues = []    row_count = df.count()        if row_count == 0:        return [{            "sheet": sheet_key,            "issue": "❌ Empty DataFrame",            "severity": "high"        }]    # Check for overall data quality    try:        # 1. Check for null values in each column        for field in df.schema.fields:            name = field.name            try:                # Handle both null and NaN values                null_count = df.filter(col(name).isNull() | isnan(col(name))).count()                null_pct = null_count / row_count if row_count > 0 else 0                                if null_pct >= threshold_null_pct:                    issues.append({                        "sheet": sheet_key,                        "column": name,                        "issue": f"⚠️ {int(null_pct * 100)}% nulls",                        "severity": "medium" if null_pct < 0.95 else "high"                    })                                # 2. Check for numeric values stored as strings                if field.dataType.simpleString() == "string":                    # Sample the first 100 rows for efficiency                    sample = df.select(col(name)).limit(100).rdd.flatMap(lambda x: [x[0]]).collect()                    numeric_count = sum(1 for val in sample if val and isinstance(val, str) and                                        val.replace(',', '').replace('.', '').replace('-', '').isdigit())                                        if numeric_count > len(sample) * 0.5:                        issues.append({                            "sheet": sheet_key,                            "column": name,                            "issue": "⚠️ Likely numeric values stored as strings",                            "severity": "low"                        })                                    except Exception as col_error:                issues.append({                    "sheet": sheet_key,                    "column": name,                    "issue": f"❌ Error checking column: {str(col_error)}",                    "severity": "medium"                })                        # 3. Check for duplicate rows        try:            dup_count = row_count - df.dropDuplicates().count()            if dup_count > 0:                issues.append({                    "sheet": sheet_key,                    "issue": f"⚠️ {dup_count} duplicate rows detected",                    "severity": "medium"                })        except Exception as e:            issues.append({                "sheet": sheet_key,                "issue": f"❌ Error checking duplicates: {str(e)}",                "severity": "low"            })                except Exception as e:        issues.append({            "sheet": sheet_key,            "issue": f"❌ Error during quality check: {str(e)}",            "severity": "high"        })            return issues

In [0]:
# Enhanced Spark DataFrame Creation with Performance Monitoring and Error Handlingfrom pyspark.sql import Rowimport jsonfrom pyspark.sql.functions import current_timestamp, lit# Track performanceperformance_monitor.start("convert_to_spark")base_path = BASE_DELTA_PATHspark_dfs = {}quality_issues_log = []for sheet_key, col_data in extracted_data_map.items():    try:        print(f"\n🧱 Converting to Spark DataFrame: {sheet_key}")        if not col_data or all(len(v) == 0 for v in col_data.values() if isinstance(v, list)):            print(f"⚠️ Skipping empty: {sheet_key}")            continue        # Find maximum length and pad columns        try:            max_len = max(len(v) for v in col_data.values() if isinstance(v, list))                        for col in list(col_data.keys()):                if isinstance(col_data[col], list):                    col_data[col] += [""] * (max_len - len(col_data[col]))                else:                    print(f"⚠️ Removing non-list column: {col}")                    del col_data[col]        except Exception as e:            print(f"⚠️ Error padding columns: {e}")            continue        # ✅ Add extra_fields if toggle is enabled        if ENABLE_EXTRA_FIELD_CAPTURE:            # fallback if not already captured            col_data["extra_fields"] = col_data.get("extra_fields", ["[]"] * max_len)        # Create pandas DataFrame        df_pd = pd.DataFrame(col_data)        file_name, sheet_name = sheet_key.split("::")        df_pd["file_name"] = file_name        df_pd["sheet_name"] = sheet_name                # Add processing timestamp        df_pd["processed_timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")        # Convert to Spark        df_spark = spark.createDataFrame(df_pd)        spark_dfs[sheet_key] = df_spark        # ✅ Quality Check        if ENABLE_QUALITY_CHECK:            performance_monitor.start(f"quality_check_{sheet_key}")            issues = flag_data_quality_issues(df_spark, sheet_key)            performance_monitor.end(f"quality_check_{sheet_key}")                        if issues:                quality_issues_log.extend(issues)                print(f"📉 Quality issues in {sheet_key}:")                for i in issues:                    print(f"  - {i['column'] if 'column' in i else 'unknown'}: {i['issue']}")        # ✅ Write to Delta if enabled        if ENABLE_DELTA_WRITE:            safe_sheet = sheet_name.replace(" ", "_").lower()            output_path = f"{base_path}/{file_name}/{safe_sheet}"            print(f"📁 Writing Delta table to: {output_path}")            success = safe_delta_write(df_spark, output_path)            if success:                print(f"✅ Delta write complete for: {sheet_key}")        print(f"✅ Spark DataFrame: {sheet_key} → {df_spark.count()} rows")    except Exception as e:        print(f"❌ Failed to convert/write {sheet_key} — {e}")performance_monitor.end("convert_to_spark", len(spark_dfs))# Save quality issues to Delta if enabledif ENABLE_QUALITY_CHECK and quality_issues_log:    try:        performance_monitor.start("save_quality_issues")                # Create DataFrame from quality issues        df_issues = spark.createDataFrame(pd.DataFrame(quality_issues_log))                # Add timestamp        df_issues = df_issues.withColumn("logged_at", current_timestamp())                # Write to Delta        if ENABLE_DELTA_WRITE:            table_name = "bdx.data_dictionary.quality_issues"            success = safe_delta_write(df_issues, table_name=table_name)                        if success:                print(f"✅ Data quality issues saved to {table_name}")                performance_monitor.end("save_quality_issues")    except Exception as e:        print(f"❌ Failed to write quality issues to Delta: {e}")# Print performance report if monitoring is enabledif ENABLE_MONITORING:    performance_monitor.print_report()